# Notebook 02 — Data Cleaning & SQLite Loading

Takes raw CSVs from Notebook 01 and:
1. Cleans and normalizes all fields
2. Tags racial equity grants via keyword matching
3. Loads into a local SQLite database
4. Documents all cleaning decisions

In [ ]:
import sys
sys.path.insert(0, '..')

import sqlite3
from pathlib import Path

import pandas as pd

from src.data_cleaning import (
    clean_grants_df,
    clean_foundations_df,
    normalize_ein,
    get_connection,
    load_table,
)

RAW  = Path('../data/raw')
PROC = Path('../data/processed')
PROC.mkdir(exist_ok=True)
DB   = Path('../data/racial_equity_grants.sqlite')

## 1  Initialize SQLite Database

In [ ]:
conn = get_connection(DB)

with open('../sql/create_tables.sql') as f:
    conn.executescript(f.read())

conn.commit()
print('Database initialized:', DB)

## 2  Clean Grants Data

**Cleaning decisions:**
- Normalize EINs to 9-digit zero-padded strings (some sources omit leading zeros)
- Drop rows where `funder_ein` is null — we cannot attribute the grant
- Drop rows where `grant_amount` is null or zero — uninformative for dollar-weighted analysis
- Standardize org names to uppercase, remove punctuation noise (preserves legal suffixes for deduplication)
- `tax_year` coerced to integer; rows with unparseable years retain null and are excluded from time-series analysis
- `is_racial_equity` flag applied via keyword regex on `grant_purpose` text (see `src/data_cleaning.py`)

In [ ]:
grants_raw = pd.read_csv(RAW / 'grants_raw.csv', dtype=str)
print(f"Raw grants: {len(grants_raw):,} rows")
grants_raw.head()

In [ ]:
grants = clean_grants_df(grants_raw)
print(f"Cleaned grants: {len(grants):,} rows")
print(f"Racial equity grants: {grants['is_racial_equity'].sum():,} ({grants['is_racial_equity'].mean():.1%})")
grants.head()

In [ ]:
# Anomaly check: grant amounts
print("Grant amount distribution:")
grants['grant_amount'].describe().apply(lambda x: f'${x:,.0f}')

In [ ]:
# Flag suspiciously large grants for review (>$500M)
outliers = grants[grants['grant_amount'] > 500_000_000]
if len(outliers):
    print(f"WARNING: {len(outliers)} grants exceed $500M — review for data errors")
    print(outliers[['funder_ein','funder_name','grant_amount','tax_year','grant_purpose']])
else:
    print("No obvious outliers found.")

In [ ]:
grants.to_csv(PROC / 'grants_clean.csv', index=False)
load_table(grants, 'grants', conn)
conn.commit()

## 3  Clean Foundations Data

In [ ]:
funders_raw = pd.read_csv(RAW / 'funders_raw.csv', dtype=str)
funders = clean_foundations_df(funders_raw)

# One row per funder (dedup keeping most recent tax year)
funders = funders.sort_values('tax_year', ascending=False).drop_duplicates('ein')
funders['source'] = 'irs_990pf'

print(f"Unique foundations: {len(funders):,}")
funders.head()

In [ ]:
funders.to_csv(PROC / 'foundations_clean.csv', index=False)
load_table(funders, 'foundations', conn)
conn.commit()

## 4  Clean Recipients Data

In [ ]:
recipients_raw = pd.read_csv(RAW / 'recipients_raw.csv', dtype=str)
recipients = recipients_raw.copy()

recipients['ein'] = recipients['ein'].apply(normalize_ein)
recipients = recipients.dropna(subset=['ein'])

# Derive NTEE major category from first character of NTEE code
recipients['ntee_major'] = recipients['ntee_code'].str[:1].str.upper()
recipients['total_revenue'] = pd.to_numeric(recipients['total_revenue'], errors='coerce')
recipients['source'] = 'propublica'

print(f"Unique recipients: {len(recipients):,}")
print(f"NTEE major categories: {recipients['ntee_major'].value_counts().to_dict()}")

In [ ]:
recipients.to_csv(PROC / 'recipients_clean.csv', index=False)
load_table(recipients, 'recipients', conn)
conn.commit()

## 5  Validation Queries

In [ ]:
checks = {
    'Total grants': 'SELECT COUNT(*) FROM grants',
    'Racial equity grants': 'SELECT COUNT(*) FROM grants WHERE is_racial_equity=1',
    'Unique funders': 'SELECT COUNT(DISTINCT funder_ein) FROM grants',
    'Total dollars ($M)': 'SELECT ROUND(SUM(grant_amount)/1e6,1) FROM grants WHERE is_racial_equity=1',
    'Foundations in DB': 'SELECT COUNT(*) FROM foundations',
    'Recipients in DB': 'SELECT COUNT(*) FROM recipients',
}

for label, sql in checks.items():
    val = conn.execute(sql).fetchone()[0]
    print(f"{label:35s} {val}")

## Summary
Cleaned data saved to `data/processed/` and loaded into `data/racial_equity_grants.sqlite`.

Proceed to **Notebook 03** for exploratory analysis.